In [ ]:
import pandas as pd
import numpy as np
import time
import unicodedata


def remove_tildes(text):
    return unicodedata.normalize("NFD", text).encode("ascii", "ignore").decode("utf-8")


pd.set_option("display.max_rows", 100)

In [2]:
cols_cie10 = ['Código', 'Descripción', 'Sección']
ruta = 'datasets/grd/CIE-10.xlsx'
cie10 = pd.read_excel(ruta, usecols=cols_cie10, dtype={col:'string' for col in cols_cie10})
cie10.rename(columns={
    'Código':'DIAGNOSTICO1',
    'Descripción':'DESCRIPCION',
    'Sección':'SECCION'
    }, inplace=True)

In [3]:
cie10.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39873 entries, 0 to 39872
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   DIAGNOSTICO1  39873 non-null  string
 1   DESCRIPCION   39873 non-null  string
 2   SECCION       39873 non-null  string
dtypes: string(3)
memory usage: 934.7 KB


In [4]:
ruta = 'datasets/estimaciones_indice_pobreza_multidimensional_comunas_2022.xlsx'
dtypes_dibat = {
    'Código':'string',
    'Porcentaje de personas en situación de pobreza multidimensional 2022':'float32'
}
cols_dibat = list(dtypes_dibat.keys())
dibat = pd.read_excel(ruta, header=2, usecols=cols_dibat, dtype=dtypes_dibat)
dibat['Código'] = dibat['Código'].str.strip().str.zfill(5)
dibat.rename(columns={
    'Código':'COD_COMUNA',
    'Porcentaje de personas en situación de pobreza multidimensional 2022':'POB_MULTIDIM'
    }, inplace=True)
dibat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 345 entries, 0 to 344
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   COD_COMUNA    345 non-null    string 
 1   POB_MULTIDIM  345 non-null    float32
dtypes: float32(1), string(1)
memory usage: 4.2 KB


In [5]:
ruta = f"datasets/base establecimientos/Base de Establecimientos 2024.xlsx"
dtypes_estab = {'Código Vigente':'string',
        'Código Región':'str',
        'Nombre Región':'category',
        'Código Comuna':'str',
        'Nombre Comuna':'category',
        'Nombre Oficial':'string',
        'Tipo de Prestador Sistema de Salud': 'string',
        'Nivel de Complejidad': 'category',
        'LATITUD      [Grados decimales]':'float32',
        'LONGITUD [Grados decimales]':'float32',
        }
cols_estab = list(dtypes_estab.keys())
estab = pd.read_excel(ruta, header=1, usecols=cols_estab, dtype=dtypes_estab, 
                                na_values={"LATITUD      [Grados decimales]": "Pendiente",
                                "LONGITUD [Grados decimales]": "Pendiente"}, decimal=',')

estab['Código Región'] = estab['Código Región'].astype('string')
estab['Código Comuna'] = estab['Código Comuna'].astype('string')
estab['Código Vigente'] = estab['Código Vigente'].astype('string')
estab.rename(columns={
        "Código Vigente":"COD_ESTABLECIMIENTO",
        'Código Región':'COD_REGION',
        'Código Comuna':'COD_COMUNA',
        'Nombre Región':'REGION',
        'Nombre Comuna':'COMUNA',
        'Nombre Oficial':'ESTABLECIMIENTO',
        'Tipo de Prestador Sistema de Salud':'PRESTADOR',
        'Nivel de Complejidad':'COMPLEJIDAD_ESTAB',
        'LATITUD      [Grados decimales]':'LATITUD',
        'LONGITUD [Grados decimales]':'LONGITUD',
        }, inplace=True)

estab['COMUNA'] = estab['COMUNA'].str.strip().str.upper().apply(remove_tildes)
estab['COD_COMUNA'] = estab['COD_COMUNA'].str.strip().str.zfill(5)
estab['PRESTADOR'] = estab['PRESTADOR'].str.strip()
estab['ESTABLECIMIENTO'] = estab['ESTABLECIMIENTO'].str.strip()
estab = estab.dropna().reset_index(drop=True)
display(estab.shape, estab.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4585 entries, 0 to 4584
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   COD_ESTABLECIMIENTO  4585 non-null   string  
 1   COD_REGION           4585 non-null   string  
 2   REGION               4585 non-null   category
 3   ESTABLECIMIENTO      4585 non-null   string  
 4   COD_COMUNA           4585 non-null   string  
 5   COMUNA               4585 non-null   object  
 6   LATITUD              4585 non-null   float32 
 7   LONGITUD             4585 non-null   float32 
 8   PRESTADOR            4585 non-null   string  
 9   COMPLEJIDAD_ESTAB    4585 non-null   category
dtypes: category(2), float32(2), object(1), string(5)
memory usage: 260.7+ KB


(4585, 10)

None

In [6]:
ruta = 'datasets/Dotacion_Camas_Hospitalarias_2025.xlsx'
dtypes_camas = {
    'Código Establecimiento':'string',
    'Área Funcional':'string',
    **{year:'str' for year in range(2019,2025)}
}
cols_camas = list(dtypes_camas.keys())
camas = pd.read_excel(ruta, header=2, dtype=dtypes_camas, usecols=lambda col: col in cols_camas)

for year in range(2019,2025):
    camas.rename(columns={year:f'{year}'}, inplace=True)

camas.rename(columns={
    'Código Establecimiento':'COD_ESTABLECIMIENTO',
    'Área Funcional':'Area'
}, inplace=True)

camas['COD_ESTABLECIMIENTO'] = camas['COD_ESTABLECIMIENTO'].ffill()
camas = camas.dropna(
    subset= ['COD_ESTABLECIMIENTO']
    ).reset_index(drop=True)

totales = camas[camas['Area'] == 'Total'].copy()

camas_criticas = [
    'Área Cuidados Intensivos Adultos',
    'Área Cuidados Intermedios Adulto',
    'Área Cuidados Intensivos Pediátrica',
    'Área Cuidados Intermedios Pediátricos',
    'Área Neonatología Cuidados Intensivos',
    'Área Neonatología Cuidados Intermedios'
]
cols_camas_str = [f'{year}' for year in range(2019,2025)]
criticas = camas[camas['Area'].isin(camas_criticas)].copy()

totales = totales.drop(columns=['Area'])
criticas = criticas.drop(columns=['Area'])

for year in range(2019,2025):
    criticas[f'{year}'] = pd.to_numeric(criticas[f'{year}'], errors='coerce')
    totales[f'{year}'] = pd.to_numeric(totales[f'{year}'], errors='coerce')
    criticas[f'{year}'] = criticas[f'{year}'].fillna(0).astype('UInt16')
    totales[f'{year}'] = totales[f'{year}'].fillna(0).astype('UInt16')

criticas = pd.DataFrame(criticas.groupby('COD_ESTABLECIMIENTO', as_index=False).sum())
criticas = criticas.melt(
    id_vars='COD_ESTABLECIMIENTO',
    value_vars=cols_camas_str,
    var_name='Year',
    value_name='CAMAS_CRITICAS'
)

totales = totales.melt(
    id_vars='COD_ESTABLECIMIENTO',
    value_vars=cols_camas_str,
    var_name='Year',
    value_name='CAMAS_TOTALES'
)
camas = pd.merge(totales, criticas, on=['COD_ESTABLECIMIENTO','Year'], how='left')
# camas = camas[(camas['CAMAS_CRITICAS'] != 0) | (camas['CAMAS_TOTALES'] != 0)].reset_index(drop=True)
camas['CAMAS_CRITICAS'] = camas['CAMAS_CRITICAS'].fillna(0)
camas['Year'] = camas['Year'].astype('int32')
camas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1182 entries, 0 to 1181
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   COD_ESTABLECIMIENTO  1182 non-null   string
 1   Year                 1182 non-null   int32 
 2   CAMAS_TOTALES        1182 non-null   UInt16
 3   CAMAS_CRITICAS       1182 non-null   UInt16
dtypes: UInt16(2), int32(1), string(1)
memory usage: 20.9 KB


In [7]:
ruta = 'datasets/Proyeccion_2002-2035_Comunas.xlsx'
dtypes_pob = {
    'Region':'string',
    'Comuna':'string',
    **{f'Poblacion {year}':'uint16' for year in range(2019,2025)}
}
cols_pob = list(dtypes_pob.keys())

poblacion = pd.read_excel(ruta, usecols=cols_pob, dtype=dtypes_pob)
poblacion = poblacion.melt(
    id_vars=['Region','Comuna'],
    value_vars=[f'Poblacion {year}' for year in range(2019,2025)],
    var_name='Year',
    value_name='Poblacion'
)

poblacion['Year'] = poblacion['Year'].str.replace('Poblacion ', '').astype('int32')

comunas = poblacion.groupby(
    by=['Region','Comuna','Year'],
    observed=True
    )['Poblacion'].sum().reset_index()
comunas.rename(columns={'Poblacion':'POB_COMUNA'}, inplace=True)

regiones = comunas.groupby(
    by=['Region','Year'],
    observed=True
    )['POB_COMUNA'].sum().reset_index()
regiones.rename(columns={'POB_COMUNA':'POB_REGION'}, inplace=True)

poblacion = pd.merge(
    comunas, 
    regiones, 
    on=['Region','Year'],
    how='left'
)
poblacion['Comuna'] = poblacion['Comuna'].str.strip().str.zfill(5)
poblacion.rename(columns={'Comuna':'COD_COMUNA', 'Region':'COD_REGION'}, inplace=True)
display(poblacion.shape, poblacion.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2076 entries, 0 to 2075
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   COD_REGION  2076 non-null   string
 1   COD_COMUNA  2076 non-null   string
 2   Year        2076 non-null   int32 
 3   POB_COMUNA  2076 non-null   uint64
 4   POB_REGION  2076 non-null   uint64
dtypes: int32(1), string(2), uint64(2)
memory usage: 73.1 KB


(2076, 5)

None

In [8]:
dtypes_grd = {'COD_HOSPITAL':"string", 
        'CIP_ENCRIPTADO':'string',
        'SEXO':"category",
        'PREVISION':"category",
        'COMUNA':"category",
        'PROVINCIA':"category",
        'TIPOALTA':"category",
        'SERVICIO_SALUD':"category",
        'TIPO_PROCEDENCIA':"category",
        'TIPO_INGRESO':"category",
        'DIAGNOSTICO1':'string',
        'IR_29301_COD_GRD':'string',
        'IR_29301_PESO':'string',
        'IR_29301_SEVERIDAD':"category",
        'IR_29301_MORTALIDAD':"category"
        }

target_cols = list(dtypes_grd.keys())
cols_fechas = ['FECHA_NACIMIENTO', 'FECHA_INGRESO', 'FECHAALTA']
target_cols.extend(cols_fechas)
config_carga = {
        "sep": "|",
        "parse_dates": cols_fechas,
        "low_memory":False,
        "usecols": target_cols,
        "on_bad_lines": "skip",
        "dtype": dtypes_grd,
}

In [9]:
ENCODINGS_TO_TRY = ['utf-8-sig', 'utf-16', 'utf-8', 'cp1252', 'latin1']

def detect_encoding(filepath):
    for enc in ENCODINGS_TO_TRY:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                f.read(8192)
            return enc
        except (UnicodeDecodeError, UnicodeError):
            continue
    return 'latin1'  # last resort, never crashes


grds = {}
for year in range(2019,2025):
    print(f'opening grd{year}...')
    params = config_carga.copy()
    cols = target_cols.copy()
    ruta = f"datasets/grd/GRD_PUBLICO_{year}.txt"
    encoding = detect_encoding(ruta)
    inicio = time.time()
    if year == 2024:
        cols[1] = 'ID_BENEFICIARIO'
        params['usecols'] = cols
    grds[f'grd{year}'] = pd.read_csv(ruta, encoding = encoding, **params)
    final = time.time()
    print(f'loaded grd{year} on {final-inicio:.3f}s\n')

opening grd2019...
loaded grd2019 on 6.523s

opening grd2020...
loaded grd2020 on 4.343s

opening grd2021...
loaded grd2021 on 4.766s

opening grd2022...
loaded grd2022 on 5.643s

opening grd2023...


/tmp/ipykernel_4027/4167503492.py:25: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  grds[f'grd{year}'] = pd.read_csv(ruta, encoding = encoding, **params)


loaded grd2023 on 6.403s

opening grd2024...
loaded grd2024 on 6.820s



In [ ]:
# se estandariza la fecha de 2023, que tiene ciertas columnas con dias primero
grds['grd2023']['FECHAALTA'] = pd.to_datetime(grds['grd2023']['FECHAALTA'], dayfirst=True, errors='coerce')
grds['grd2023']['FECHA_INGRESO'] = pd.to_datetime(grds['grd2023']['FECHA_INGRESO'], dayfirst=True, errors='coerce')
grds['grd2024'].rename(columns={"ID_BENEFICIARIO":"CIP_ENCRIPTADO"}, inplace=True)
grd = pd.concat(grds.values(), ignore_index=True)

# se pasan las columnas a dtype datetime64
for col in cols_fechas:
    grd[col] = pd.to_datetime(grd[col], errors='coerce')

# se lee el rut encriptado como dtype string, para que no se considere un
# numero y aparezca en las estadisticas erroneamente
grd['CIP_ENCRIPTADO'] = grd['CIP_ENCRIPTADO'].astype('string')

# se cambia la coma por punto para interpretar como float
grd['IR_29301_PESO'] = grd.IR_29301_PESO.str.replace(',','.', regex=False)
# los que NO tienen punto
dec_coma = ~grd['IR_29301_PESO'].str.contains('.', regex=False)
# todos como float
floats = pd.to_numeric(grd['IR_29301_PESO'], errors='coerce')
# grd['IR_29301_PESO'] = grd.IR_29301_PESO.str.replace('.','')
powers = np.maximum(grd['IR_29301_PESO'].str.len() - 1,1)
# se divide solo donde no hay punto, por error de tipeo
grd['IR_29301_PESO'] = np.where(dec_coma, floats / (10 ** powers), floats)

# columnas categoricas, que guardan valores unicos, o categorias de pandas
categ_cols = ['SEXO', 'PREVISION', 'TIPO_PROCEDENCIA', 'TIPO_INGRESO', 'IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD',
            'TIPOALTA', 'PROVINCIA', 'COMUNA','SERVICIO_SALUD']

# se eliminan nulos tras estandarizar la mayoria de columnas
grd = grd.dropna(ignore_index=True)
# se transforman las columnas anteriores a categoricas
grd = grd.astype({col: 'category' for col in categ_cols})
# se reescriben ciertos nombres de columna para calzar con los merges proximos
grd.rename(columns={'COMUNA':'COMUNA_REGISTRADA', 'COD_HOSPITAL':'COD_ESTABLECIMIENTO'}, inplace=True)

# se consideran casos de peso > 2 y severidad 2 o 3, quedando con 6.4% del dataset
grd = grd[grd['IR_29301_PESO'] > 2]
grd = grd[(grd['IR_29301_SEVERIDAD'] == '2') | (grd['IR_29301_SEVERIDAD'] == '3')]
grd = grd.reset_index(drop=True)

# se estandarizan las comunas para que la registrada y del establecimiento
# esten escritas de la misma manera y puedan compararse
grd['COMUNA_REGISTRADA'] = grd['COMUNA_REGISTRADA'].str.strip().str.upper().apply(remove_tildes)
grd['COMUNA_REGISTRADA'] = grd['COMUNA_REGISTRADA'].str.replace('COIHAIQUE','COYHAIQUE').astype('category')
# se crean variables clave, edad y dias estancia
grd['EDAD'] = ((grd['FECHA_INGRESO'] - grd['FECHA_NACIMIENTO']).dt.days / 365.25).round(1)
grd['DIAS_ESTANCIA'] = (grd['FECHAALTA'] - grd['FECHA_INGRESO']).dt.days
grd['IR_29301_SEVERIDAD'] = grd.IR_29301_SEVERIDAD.astype('string').astype('category')

In [ ]:
# se realizan todos los merge para los que se preparo la base de grd
# merge con cie10
grd_merge = pd.merge(grd, cie10, on='DIAGNOSTICO1', how='left')
# merge con base establecimientos
grd_merge = pd.merge(grd_merge, estab, on='COD_ESTABLECIMIENTO',how='left')
# se filtran datos para que solo sean con fecha ingreso >= 2019
grd_merge['Year'] = grd_merge['FECHA_INGRESO'].dt.year
grd_merge['Year'] = grd_merge['Year'].astype('int32')
grd_merge = grd_merge[grd_merge['Year'] >= 2019]
grd_merge.reset_index(drop=True, inplace=True)
# merge con proyeccion poblacion del ine
grd_merge = pd.merge(grd_merge, poblacion, on=['COD_REGION','COD_COMUNA','Year'], how='left')
# merge con camas hospitalarias por establecimiento
grd_merge = pd.merge(grd_merge, camas, on=['COD_ESTABLECIMIENTO','Year'], how='left')
# merge con indice pobreza multidimensional de la DIBAT
grd_merge = pd.merge(grd_merge, dibat, on='COD_COMUNA', how='left')
grd_merge.rename(columns={'COMUNA':'COMUNA_ESTAB'}, inplace=True)

In [ ]:
# se transforman los dtypes de las nuevas columnas de poblacion
grd_merge['POB_COMUNA'] = grd_merge['POB_COMUNA'].astype('int32')
grd_merge['POB_REGION'] = grd_merge['POB_REGION'].astype('int32')
# se dropean nulos tras merge
grd_merge = grd_merge.dropna().reset_index(drop=True)
categ_cols = ['COD_ESTABLECIMIENTO', 'COD_COMUNA','COD_REGION','Year']
for col in categ_cols:
    grd_merge[col] = grd_merge[col].astype('category')
display(grd_merge.shape, grd_merge.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367895 entries, 0 to 367894
Data columns (total 37 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   COD_ESTABLECIMIENTO  367895 non-null  category      
 1   CIP_ENCRIPTADO       367895 non-null  string        
 2   SEXO                 367895 non-null  category      
 3   FECHA_NACIMIENTO     367895 non-null  datetime64[ns]
 4   PROVINCIA            367895 non-null  category      
 5   COMUNA_REGISTRADA    367895 non-null  category      
 6   PREVISION            367895 non-null  category      
 7   SERVICIO_SALUD       367895 non-null  category      
 8   TIPO_PROCEDENCIA     367895 non-null  category      
 9   TIPO_INGRESO         367895 non-null  category      
 10  FECHA_INGRESO        367895 non-null  datetime64[ns]
 11  FECHAALTA            367895 non-null  datetime64[ns]
 12  TIPOALTA             367895 non-null  category      
 13  DIAGNOSTICO1  

(367895, 37)

None

In [15]:
grd_merge.to_parquet("grd_procesado.parquet")

In [16]:
grd_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367895 entries, 0 to 367894
Data columns (total 37 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   COD_ESTABLECIMIENTO  367895 non-null  category      
 1   CIP_ENCRIPTADO       367895 non-null  string        
 2   SEXO                 367895 non-null  category      
 3   FECHA_NACIMIENTO     367895 non-null  datetime64[ns]
 4   PROVINCIA            367895 non-null  category      
 5   COMUNA_REGISTRADA    367895 non-null  category      
 6   PREVISION            367895 non-null  category      
 7   SERVICIO_SALUD       367895 non-null  category      
 8   TIPO_PROCEDENCIA     367895 non-null  category      
 9   TIPO_INGRESO         367895 non-null  category      
 10  FECHA_INGRESO        367895 non-null  datetime64[ns]
 11  FECHAALTA            367895 non-null  datetime64[ns]
 12  TIPOALTA             367895 non-null  category      
 13  DIAGNOSTICO1  